In [1]:
#1. 数据加载与结构解析
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

# 加载数据
df = pd.read_excel('AOactivity3.xlsx')
print("数据集基本信息:", df.shape)

# 列名适配
compound_id_col = "chemical ID" if "chemical ID" in df.columns else "compound_id"
ao_label_col = "AO1065"

# 因果链节点
causal_chain_nodes = [
    "MIE2293", "KE1076", "KE2294", "KE2251",
    "MIE1046", "KE1047", "KE1050", "KE1972", "KE1973",
    "MIE1065", "KE1985", "KE2137", "KE2402",
    "MIE2155", "MIE2250",
    "KE530", "KE531", "KE1695", "KE2303",
    "KE1075"
]

# 筛选特征
feature_cols = [col for col in causal_chain_nodes if col in df.columns and col != ao_label_col]
print(f"特征数: {len(feature_cols)}")
df[feature_cols + [ao_label_col]] = df[feature_cols + [ao_label_col]].fillna(0)

# 2. 图结构 —— 有向→无向 + 自环边，不去重边
causal_edges = [
    ("MIE2293", "KE1076"), ("KE1076", "KE2294"), ("KE2294", "KE2251"),
    ("MIE1046", "KE1047"), ("KE1047", "KE1050"), ("KE1050", "KE1972"), ("KE1972", "KE1973"), ("KE1973", "KE1076"),
    ("MIE1065", "KE1985"), ("KE1985", "KE1047"), ("KE1047", "KE2137"), ("KE2137", "KE2402"), ("KE2402", "KE2294"),
    ("MIE2155", "KE2251"), ("MIE2250", "KE2251"),
    ("KE530", "KE531"), ("KE531", "KE1695"), ("KE1695", "KE2303"), ("KE2303", "KE2251"),
    ("KE1075", "KE1076")
]

# ========== 图结构：无向图 + 自环边，不去重 ==========
# 1. 生成反向边，转为无向图
undirected_edges = []
for s, d in causal_edges:
    undirected_edges.append((s, d))
    undirected_edges.append((d, s))  # 反向边

# 2. 添加自环边
all_nodes = list(set(n for e in undirected_edges for n in e))
for node in all_nodes:
    undirected_edges.append((node, node))  # 自环

# 🔴 这里不去重！
# undirected_edges = list(set(undirected_edges))

# 构建最终边索引
node_id_map = {n:i for i,n in enumerate(all_nodes)}
edge_index = torch.tensor([[node_id_map[s], node_id_map[d]] for s,d in undirected_edges], dtype=torch.long).T

# ========== 节点特征：度数 + 节点类型 ==========
node_degree = {node: 0 for node in all_nodes}
for s, d in undirected_edges:
    if s != d:
        node_degree[s] += 1

node_type = {node: 1 if node.startswith("MIE") else 0 for node in all_nodes}

degree_tensor = torch.tensor([node_degree[node] for node in all_nodes], dtype=torch.float32).unsqueeze(1)
type_tensor = torch.tensor([node_type[node] for node in all_nodes], dtype=torch.float32).unsqueeze(1)

# 3. 数据集类
class AO1065GraphDataset(Dataset):
    def __init__(self, df, feature_cols, ao_label_col, compound_id_col, node_id_map):
        super().__init__()
        self.df = df
        self.feature_cols = feature_cols
        self.ao_label_col = ao_label_col
        self.compound_id_col = compound_id_col
        self.node_id_map = node_id_map
        self.num_nodes = len(node_id_map)

    def len(self): return len(self.df)
    
    def get(self, idx):
        row = self.df.iloc[idx]
        x_activate = torch.zeros((self.num_nodes, 1), dtype=torch.float32)
        for col in self.feature_cols:
            x_activate[self.node_id_map[col], 0] = row[col]
        
        x = torch.cat([x_activate, degree_tensor, type_tensor], dim=1)
        y = torch.tensor([row[self.ao_label_col]], dtype=torch.float32)
        return Data(x=x, edge_index=edge_index, y=y, compound_id=row[self.compound_id_col])

# 数据集划分
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df[ao_label_col]
)

train_dataset = AO1065GraphDataset(df.loc[train_idx].reset_index(drop=True), feature_cols, ao_label_col, compound_id_col, node_id_map)
test_dataset  = AO1065GraphDataset(df.loc[test_idx].reset_index(drop=True),  feature_cols, ao_label_col, compound_id_col, node_id_map)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

# ==========================================
# GraphSAGE 精调版
# ==========================================
class GraphSAGE_Tuned(torch.nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.15):
        super().__init__()
        self.conv1 = SAGEConv(3, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.conv3 = SAGEConv(hidden_dim, hidden_dim)
        
        self.bn1 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn2 = torch.nn.BatchNorm1d(hidden_dim)
        self.bn3 = torch.nn.BatchNorm1d(hidden_dim)
        
        self.lin1 = torch.nn.Linear(hidden_dim, 64)
        self.lin2 = torch.nn.Linear(64, 1)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)

        res = x
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = x + res

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.lin2(x)
        return x

# ====================== 优化器与学习率 ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GraphSAGE_Tuned(hidden_dim=128).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0], device=device))

# 训练函数
def train_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y.unsqueeze(1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.8)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_dataset)

# 评估函数
@torch.no_grad()
def evaluate():
    model.eval()
    y_true, y_prob, ids = [], [], []
    for batch in test_loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        prob = torch.sigmoid(logits).cpu().numpy()
        y_true.extend(batch.y.cpu().numpy())
        y_prob.extend(prob)
        ids.extend(batch.compound_id)
    
    y_true = np.array(y_true).flatten()
    y_prob = np.array(y_prob).flatten()

    best_f1 = 0
    best_t = 0.5
    for t in np.linspace(0.30, 0.70, 71):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    acc = accuracy_score(y_true, (y_prob >= best_t).astype(int))
    auc = roc_auc_score(y_true, y_prob)
    return acc, auc, best_f1, best_t, y_true, y_prob, ids

# 训练循环
best_auc = 0
patience = 0
early_stop_patience = 70

print("\n🚀 GraphSAGE 增强版训练开始")
for epoch in range(1, 501):
    train_loss = train_epoch()
    acc, auc, f1, best_t, _, _, _ = evaluate()
    
    scheduler.step()
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "gcntry10.pth")
        patience = 0
    else:
        patience += 1
        if patience >= early_stop_patience:
            print(f"⏹️ 早停 Epoch {epoch}")
            break
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss {train_loss:.4f} | Acc {acc:.4f} | AUC {auc:.4f} | F1 {f1:.4f}")

# 最终结果
model.load_state_dict(torch.load("gcntry10.pth"))
acc, auc, f1, best_t, y_true, y_prob, ids = evaluate()

print("\n" + "="*60)
print("✅ GraphSAGE 增强版最终结果")
print(f"Acc: {acc:.4f} | AUC: {auc:.4f} | F1 {f1:.4f} | 最优阈值: {best_t:.2f}")
print("="*60)

pd.DataFrame({
    '化合物ID': ids,
    '真实标签': y_true.astype(int),
    '预测概率': np.round(y_prob,4),
    '预测类别': (y_prob>=best_t).astype(int)
}).to_excel('AO1065_GraphSAGE_ENHANCED.xlsx', index=False)

D:\anaconda\envs\jupyter-clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据集基本信息: (1765, 24)
特征数: 20

🚀 GraphSAGE 增强版训练开始
Epoch 010 | Loss 0.4045 | Acc 0.8045 | AUC 0.8923 | F1 0.7890
Epoch 020 | Loss 0.3575 | Acc 0.8669 | AUC 0.9063 | F1 0.8469
Epoch 030 | Loss 0.3713 | Acc 0.8385 | AUC 0.8927 | F1 0.7849
Epoch 040 | Loss 0.3351 | Acc 0.8725 | AUC 0.9118 | F1 0.8399
Epoch 050 | Loss 0.3075 | Acc 0.8980 | AUC 0.9203 | F1 0.8750
Epoch 060 | Loss 0.2957 | Acc 0.8952 | AUC 0.9232 | F1 0.8702
Epoch 070 | Loss 0.3275 | Acc 0.8357 | AUC 0.9117 | F1 0.8176
Epoch 080 | Loss 0.3204 | Acc 0.8612 | AUC 0.9265 | F1 0.8393
Epoch 090 | Loss 0.3286 | Acc 0.8810 | AUC 0.9185 | F1 0.8571
Epoch 100 | Loss 0.3028 | Acc 0.8810 | AUC 0.9230 | F1 0.8552
Epoch 110 | Loss 0.2824 | Acc 0.8300 | AUC 0.9274 | F1 0.8125
Epoch 120 | Loss 0.2476 | Acc 0.9037 | AUC 0.9274 | F1 0.8777
Epoch 130 | Loss 0.2584 | Acc 0.8895 | AUC 0.9298 | F1 0.8660
Epoch 140 | Loss 0.2528 | Acc 0.8924 | AUC 0.9287 | F1 0.8681
Epoch 150 | Loss 0.2996 | Acc 0.8442 | AUC 0.9092 | F1 0.8029
Epoch 160 | Loss 0.30

In [ ]:
# ==========================================
# 🔍 特征重要性 & 节点重要性分析（最终完美版）
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 1. 加载最优模型
model.load_state_dict(torch.load("gcntry10.pth"))
model.eval()

# 2. 特征名称
feature_names = [
    "节点激活值特征",
    "节点度中心性",
    "节点类型(MIE=1)"
]

# 3. 计算特征重要性（逐图处理，解决批次拼接问题）
def compute_feature_importance():
    model.eval()
    feat_importance = torch.zeros(3, device=device)
    node_importance = torch.zeros(len(all_nodes), device=device)
    total_graphs = 0

    with torch.enable_grad():
        for batch in train_loader:
            batch = batch.to(device)
            num_graphs = batch.num_graphs
            total_graphs += num_graphs

            # 拆分成单张图逐张计算（避免拼接后维度爆炸）
            for i in range(num_graphs):
                g = batch.get_example(i)
                g.x.requires_grad_(True)

                logits = model(g.x, g.edge_index, g.batch)
                prob = torch.sigmoid(logits)

                grad = torch.autograd.grad(
                    outputs=prob.sum(), inputs=g.x, retain_graph=False
                )[0]

                feat_importance += grad.abs().sum(dim=0)
                node_importance += grad.abs().sum(dim=1)

    # 归一化
    feat_importance = feat_importance / feat_importance.sum()
    node_importance = node_importance / node_importance.sum()
    return feat_importance.detach().cpu().numpy(), node_importance.detach().cpu().numpy()

# 4. 运行
feat_imp, node_imp = compute_feature_importance()

# 5. 节点重要性表格
id2node = {i: node for node, i in node_id_map.items()}
node_importance_df = pd.DataFrame({
    "节点名称": [id2node[i] for i in range(len(all_nodes))],
    "节点类型": ["MIE" if n.startswith("MIE") else "KE" for n in id2node.values()],
    "重要性分数": np.round(node_imp, 6)
}).sort_values("重要性分数", ascending=False).reset_index(drop=True)

# 6. 输出结果
print("\n" + "="*80)
print("📊 模型输入特征重要性")
print("="*80)
for n, v in zip(feature_names, feat_imp):
    print(f"{n:15s} : {v:.4f}")

print("\n" + "="*80)
print("🌟 AO1065 节点重要性 Top15")
print("="*80)
print(node_importance_df.head(15).to_string(index=False))

# 保存
node_importance_df.to_excel("AO1065_节点重要性排序.xlsx", index=False)

# 绘图：特征重要性
plt.figure(figsize=(8, 5))
sns.barplot(x=feature_names, y=feat_imp, palette='viridis')
plt.title("GraphSAGE 输入特征重要性", fontsize=14)
plt.ylabel("归一化重要性")
plt.tight_layout()
plt.savefig("特征重要性.png", dpi=300)
plt.show()

# 绘图：Top15 节点
top15 = node_importance_df.head(15)
plt.figure(figsize=(12, 6))
sns.barplot(x="节点名称", y="重要性分数", hue="节点类型", data=top15, palette='coolwarm')
plt.title("AO1065 因果链 Top15 重要节点", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("Top15节点重要性.png", dpi=300)
plt.show()

print("\n✅ 分析完成！已生成 Excel + 两张图片")